In [ ]:
import time
import os
import pandas as pd
import numpy as np
import cv2

from beamngpy import BeamNGpy, Scenario, Vehicle
from beamngpy.sensors import AdvancedIMU, Electrics, Camera

# ================= CONFIGURATION =================
BASE_FOLDER = "../data"
RUN_FOLDER_NAME = "test_run_east_coast_highway_trip_4"

CSV_FILENAME = "sensor_data.csv"
VIDEO_FILENAME = "dashcam.mp4"

VIDEO_FPS = 30.0
VIDEO_RES = (1280, 720)

# HIGHWAY COORDINATES
SPAWN_POS = (-852.0, -517.4, 20.2)
SPAWN_ROT = (0, 0, 0.38, 0.92) 
# =================================================

def get_imu_values(data):
    """
    Extracts ax, ay, az, wx, wy, wz from IMU data.
    PRIORITIZES 'accSmooth' over 'accRaw' to avoid physics spikes.
    """
    if not data:
        return 0, 0, 0, 0, 0, 0

    # 1. Determine which sample to use (handle buffer)
    sample = data
    if 'accSmooth' not in data and 'accRaw' not in data and 'angVel' not in data:
        if len(data) > 0:
            # If data is a dict of samples {time: sample, ...}
            first_val = next(iter(data.values()))
            if isinstance(first_val, dict):
                # Find the latest sample by time key
                try:
                    latest_key = max(data.keys())
                    sample = data[latest_key]
                except:
                    sample = first_val

    # 2. Extract Acceleration (Use Smooth first!)
    # accSmooth filters out high-freq noise/vibration
    acc = sample.get('accSmooth') or sample.get('accRaw') or sample.get('acc') or sample.get('accel')
    
    # 3. Extract Angular Velocity
    gyro = sample.get('angVelSmooth') or sample.get('angVel') or sample.get('gyro')

    # 4. Parse values
    ax, ay, az = 0, 0, 0
    if acc:
        if isinstance(acc, (list, tuple, np.ndarray)) and len(acc) >= 3:
            ax, ay, az = acc[0], acc[1], acc[2]
        elif isinstance(acc, dict):
            ax = acc.get('x', 0)
            ay = acc.get('y', 0)
            az = acc.get('z', 0)

    wx, wy, wz = 0, 0, 0
    if gyro:
        if isinstance(gyro, (list, tuple, np.ndarray)) and len(gyro) >= 3:
            wx, wy, wz = gyro[0], gyro[1], gyro[2]
        elif isinstance(gyro, dict):
            wx = gyro.get('x', 0)
            wy = gyro.get('y', 0)
            wz = gyro.get('z', 0)

    return ax, ay, az, wx, wy, wz

def main():
    # 1. Folder Setup
    run_path = os.path.join(BASE_FOLDER, RUN_FOLDER_NAME)
    os.makedirs(run_path, exist_ok=True)

    # 2. BeamNG Init
    print("Initializing BeamNG...")
    bng = BeamNGpy(
        host='localhost',
        port=64256,
        home=r"C:\BeamNG\BeamNG.tech.v0.38.3.0",
        user=r"C:\Users\nishk\AppData\Local\BeamNG\BeamNG.tech"
    )

    bng.open(launch=True)

    # 3. Scenario Setup
    scenario = Scenario('east_coast_usa', 'highway_drive')

    vehicle = Vehicle(
        'ego_vehicle', 
        model='vivace', 
        partConfig='vehicles/vivace/tograc_110_m.pc', 
        license='DATA_COL'
    )

    scenario.add_vehicle(
        vehicle,
        pos=SPAWN_POS,
        rot_quat=SPAWN_ROT
    )

    scenario.make(bng)
    bng.scenario.load(scenario)
    bng.scenario.start()

    print("Scenario started. Waiting 2 seconds...")
    time.sleep(2)

    # 4. Sensors
    imu = AdvancedIMU(
        'imu', bng, vehicle,
        pos=(0, -0.3, 0.8),
        physics_update_time=0.01
    )

    dashcam = Camera(
        'dashcam', bng, vehicle,
        pos=(0, -2, 2),
        dir=(0, -1, -0.3),
        up=(0, 0, 1),
        resolution=VIDEO_RES,
        field_of_view_y=70,
        requested_update_time=1.0 / VIDEO_FPS,
        is_using_shared_memory=True,
        is_render_colours=True,
        is_render_depth=False,
        is_render_annotations=False
    )

    electrics = Electrics()
    vehicle.sensors.attach('electrics', electrics)

    # 5. Video Writer
    video_path = os.path.join(run_path, VIDEO_FILENAME)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(video_path, fourcc, VIDEO_FPS, VIDEO_RES)

    print("\n" + "="*50)
    print("  RECORDING STARTED")
    print("  CLICK VIDEO WINDOW AND PRESS 'q' TO STOP.")
    print("="*50)

    data_log = []
    sample_count = 0
    first_frame = True

    try:
        while True:
            loop_start = time.time()

            # Poll Sensors
            vehicle.sensors.poll() 
            imu_data = imu.poll()
            cam_data = dashcam.poll()

            if first_frame:
                print(f"[DEBUG] First IMU Data: {imu_data}")
                first_frame = False

            # Get Values (Using Smooth)
            ax, ay, az, wx, wy, wz = get_imu_values(imu_data)
            
            # Get Speed in m/s (Native unit)
            speed_ms = 0.0
            if 'wheelspeed' in electrics:
                speed_ms = electrics['wheelspeed'] # Native is m/s
            
            # Log Data
            data_log.append({
                'sample': sample_count,
                'timestamp': time.time(),
                'ax': ax, 'ay': ay, 'az': az,
                'wx': wx, 'wy': wy, 'wz': wz,
                'speed': speed_ms
            })

            # Video Frame
            if 'colour' in cam_data:
                pil_img = cam_data['colour'].convert('RGB')
                frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

                cv2.putText(frame, f"REC | Sample: {sample_count}", (30, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
                
                # Display m/s on screen
                cv2.putText(frame, f"Speed: {speed_ms:.1f} m/s", (30, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
                
                cv2.putText(frame, f"AZ: {az:.2f}", (30, 130),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                video_writer.write(frame)
                cv2.imshow("Dashcam Feed", frame)

            sample_count += 1

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("\nStop signal received.")
                break

            elapsed = time.time() - loop_start
            time.sleep(max(0, 0.01 - elapsed))

    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()

    finally:
        if 'video_writer' in locals():
            video_writer.release()
        cv2.destroyAllWindows()

        # Save CSV
        df = pd.DataFrame(data_log)
        csv_path = os.path.join(run_path, CSV_FILENAME)
        df.to_csv(csv_path, index=False)
        
        print("\n" + "="*50)
        print("DONE")
        print(f"CSV Saved: {csv_path} ({len(df)} rows)")
        print(f"Video Saved: {video_path}")
        print("="*50)
        
        bng.close()

if __name__ == '__main__':
    main()

Initializing BeamNG...
Physics paused. Entering deterministic stepping mode.
Sampling at 100 Hz...
Captured 0 samples (0.01s sim time)
Captured 100 samples (1.01s sim time)
Saved 129 samples to ../data\test_run_deterministic_100hz\sensor_data_100hz.csv
